# Transfer Attack (Black-Box)

## Addestramento MobileNetV2 + Transfer Attack da ResNet18

Questo notebook:
1. Addestra MobileNetV2 sullo stesso dataset di ResNet18 (secondo modello per il black-box)
2. Valuta la clean accuracy di entrambi i modelli sul test set
3. Genera adversarial examples su ResNet18 (source) con FGSM, PGD e DeepFool
4. Trasferisce quegli stessi esempi su MobileNetV2 (target) senza accedere ai suoi gradienti

## 1. Setup e Import

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path('..')
BATCH_SIZE = 32
IMG_SIZE = 224
N_CLASSES = 5

# ResNet18 — modello sorgente (già addestrato)
source_model = models.resnet18(pretrained=True)
source_model.fc = nn.Linear(in_features=512, out_features=N_CLASSES)
source_model = source_model.to(device)
source_model.load_state_dict(torch.load(ROOT / "models" / "best_resnet18.pt"))
source_model.eval()

NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor()
])

dataset_path = ROOT / "data" / "fruits-classification-stratified"

train_dataset = ImageFolder(root=dataset_path / 'train', transform=TRAIN_TRANSFORMS)
val_dataset   = ImageFolder(root=dataset_path / 'valid', transform=TRANSFORMS)
test_dataset  = ImageFolder(root=dataset_path / 'test',  transform=TRANSFORMS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

class_names = test_dataset.classes

results_path = ROOT / "results"
results_path.mkdir(exist_ok=True)

print(f"Device: {device}")
print(f"Classi: {class_names}")
print("Setup completato")


## 2. Addestramento MobileNetV2

MobileNetV2 è un'architettura leggera e diversa da ResNet18, scelta come secondo modello per il black-box transfer attack.

Fine-tuning identico a ResNet18:
- stesso dataset e stesso split
- stessa NORMALIZE dentro il forward pass (immagini in [0,1])
- Adam optimizer, lr=0.0001, early stopping patience=20, max 50 epoche
- il classificatore finale `classifier[-1]` viene sostituito con N_CLASSES output

In [ ]:
target_model = models.mobilenet_v2(pretrained=True)
target_model.classifier[-1] = nn.Linear(in_features=target_model.classifier[-1].in_features, out_features=N_CLASSES)
target_model = target_model.to(device)

loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(target_model.parameters(), lr=0.0001)

n_epochs        = 50
patience        = 20
patience_counter = 0
best_val_loss   = float('inf')
model_save_path = ROOT / "models" / "best_mobilenet_v2.pt"

metrics = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

start_time_tot = time.time()

for epoch in range(n_epochs):
    # --- Training ---
    target_model.train()
    tot_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs} [train]"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = target_model(NORMALIZE(images))
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        tot_loss += loss.item()
        correct  += (torch.argmax(outputs, dim=1) == labels).sum().item()
        total    += labels.size(0)

    train_loss = tot_loss / len(train_loader)
    train_acc  = correct / total

    # --- Validazione ---
    target_model.eval()
    val_loss_sum, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{n_epochs} [val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = target_model(NORMALIZE(images))
            loss    = loss_fn(outputs, labels)
            val_loss_sum += loss.item()
            val_correct  += (torch.argmax(outputs, dim=1) == labels).sum().item()
            val_total    += labels.size(0)

    val_loss = val_loss_sum / len(val_loader)
    val_acc  = val_correct  / val_total

    metrics['train_loss'].append(train_loss)
    metrics['train_acc'].append(train_acc)
    metrics['val_loss'].append(val_loss)
    metrics['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(target_model.state_dict(), model_save_path)
        print("  Nuovo migliore modello salvato")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  Early Stopping alla epoca {epoch+1}")
            break

tot_time = time.time() - start_time_tot
print(f"\nTraining completato in {tot_time/60:.1f} min")
print(f"Miglior val loss: {best_val_loss:.4f}")


## 3. Valutazione clean — ResNet18 vs MobileNetV2

Verifica la clean accuracy di entrambi i modelli sul test set prima di lanciare gli attacchi.
La formula del Transfer ASR usa solo le immagini classificate correttamente da **entrambi** i modelli come denominatore, quindi questa baseline è fondamentale.

In [ ]:
# carica il miglior checkpoint di MobileNetV2
target_model.load_state_dict(torch.load(model_save_path))
target_model.eval()
source_model.eval()

source_correct = 0
target_correct = 0
total = 0

all_source_preds_clean = []
all_target_preds_clean = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Valutazione clean"):
        images, labels = images.to(device), labels.to(device)

        source_preds = torch.argmax(source_model(NORMALIZE(images)), dim=1)
        target_preds = torch.argmax(target_model(NORMALIZE(images)), dim=1)

        source_correct += (source_preds == labels).sum().item()
        target_correct += (target_preds == labels).sum().item()
        total          += labels.size(0)

        all_source_preds_clean.extend(source_preds.cpu().numpy())
        all_target_preds_clean.extend(target_preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

source_clean_acc = source_correct / total
target_clean_acc = target_correct / total

print(f"ResNet18   — Clean Accuracy: {source_clean_acc:.4f} ({source_clean_acc*100:.2f}%)")
print(f"MobileNetV2 — Clean Accuracy: {target_clean_acc:.4f} ({target_clean_acc*100:.2f}%)")

# salva metriche clean di MobileNetV2
mobilenet_summary = pd.DataFrame([{
    'model': 'MobileNetV2',
    'clean_accuracy': target_clean_acc
}])
mobilenet_summary.to_csv(results_path / "MobileNetV2_metrics_summary.csv", index=False)

# confusion matrix MobileNetV2
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, preds, title, cmap in [
    (axes[0], all_source_preds_clean, "ResNet18 — Clean", "Blues"),
    (axes[1], all_target_preds_clean, "MobileNetV2 — Clean", "Greens"),
]:
    cm = confusion_matrix(all_labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.savefig(results_path / "MobileNetV2_confusion_matrix.png", dpi=120, bbox_inches='tight')
plt.show()


## 4. Funzioni di attacco (riuso identico dai notebook precedenti)

Le funzioni sono copiate 1:1 da `02_fgsm_attacks.ipynb`, `03_pgd_attacks.ipynb` e `04_df_attacks.ipynb`.
Nessuna modifica: stessa firma, stessa logica, stesso uso di `NORMALIZE` nel forward pass.

In [ ]:
def fgsm_attack(model, images, labels, epsilon, loss_fn):
    """
    Esegue un attacco FGSM su un batch di immagini
    (copia identica da 02_fgsm_attacks.ipynb)
    """
    images.requires_grad = True

    outputs = model(NORMALIZE(images))
    loss = loss_fn(outputs, labels)

    model.zero_grad()
    loss.backward()

    gradient_sign = images.grad.sign()
    perturbation  = epsilon * gradient_sign
    adv_images    = images + perturbation
    adv_images    = torch.clamp(adv_images, 0, 1).detach()
    images.requires_grad = False

    perturbation = adv_images - images

    return adv_images, perturbation.detach()


def pgd_attack(model, images, labels, epsilon, alpha, num_steps, loss_fn):
    """
    Esegue un attacco PGD su un batch di immagini
    (copia identica da 03_pgd_attacks.ipynb)
    """
    adv_images = images.clone().detach()
    adv_images = adv_images + torch.empty_like(adv_images).uniform_(-epsilon, epsilon)
    adv_images = torch.clamp(adv_images, 0, 1).detach()

    for _ in range(num_steps):
        adv_images.requires_grad = True
        outputs = model(NORMALIZE(adv_images))
        loss    = loss_fn(outputs, labels)
        model.zero_grad()
        loss.backward()

        gradient_sign = adv_images.grad.sign()
        adv_images    = adv_images.detach() + alpha * gradient_sign
        perturbation  = torch.clamp(adv_images - images, -epsilon, epsilon)
        adv_images    = torch.clamp(images + perturbation, 0, 1).detach()

    perturbation = adv_images - images

    return adv_images, perturbation.detach()


def deepfool_attack(model, images, num_classes, overshoot, max_iter):
    """
    Esegue un attacco DeepFool su un batch di immagini
    (copia identica da 04_df_attacks.ipynb)
    """
    adv_images = images.clone().detach()

    for image_id in range(images.size(0)):
        original_image = images[image_id:image_id+1].clone().detach()

        with torch.no_grad():
            outputs      = model(NORMALIZE(original_image))
            pred_original = torch.argmax(outputs, dim=1).item()

        total_perturbation = torch.zeros_like(original_image)

        for _ in range(max_iter):
            current_image = torch.clamp(original_image + total_perturbation, 0, 1).detach()
            current_image.requires_grad_(True)

            outputs      = model(NORMALIZE(current_image))
            pred_current = torch.argmax(outputs, dim=1).item()

            if pred_current != pred_original:
                break

            gradients = []
            for class_id in range(num_classes):
                model.zero_grad()
                if current_image.grad is not None:
                    current_image.grad.zero_()
                outputs[0, class_id].backward(retain_graph=True)
                gradients.append(current_image.grad.clone().detach())

            original_gradient = gradients[pred_original]
            original_score    = outputs[0, pred_original].detach()

            min_distance = float('inf')
            best_step    = torch.zeros_like(original_image)

            for class_id in range(num_classes):
                if class_id == pred_original:
                    continue
                gradient_difference = gradients[class_id] - original_gradient
                score_difference    = outputs[0, class_id].detach() - original_score
                gradient_norm       = gradient_difference.norm()
                if gradient_norm.item() == 0:
                    continue
                distance = torch.abs(score_difference) / gradient_norm
                if distance.item() < min_distance:
                    min_distance = distance.item()
                    best_step = (torch.abs(score_difference) / gradient_norm.pow(2)) * gradient_difference

            total_perturbation = total_perturbation + best_step

        adv_image = torch.clamp(original_image + (1 + overshoot) * total_perturbation, 0, 1).detach()
        adv_images[image_id:image_id+1] = adv_image

    perturbation = adv_images - images

    return adv_images, perturbation.detach()


print("Funzioni FGSM, PGD e DeepFool pronte")


## 5. Transfer Attack

**Logica black-box:**
- Gli adversarial examples vengono generati sul modello **sorgente** (ResNet18) — qui i gradienti servono
- Gli stessi esempi vengono valutati sul modello **target** (MobileNetV2) — solo forward pass, `torch.no_grad()`
- Il target non vede mai i gradienti: questo simula il setting black-box reale

**Transfer ASR** — formula corretta:
```
both_correct    = (source_clean_preds == labels) & (target_clean_preds == labels)
transfer_success = both_correct & (target_adv_preds != labels)
transfer_asr    = transfer_success / both_correct
```
Usa `both_correct` come denominatore per non contare come successo immagini già sbagliate dal target.

In [ ]:
epsilons_transfer = [0.001, 0.003, 0.005, 0.01]
num_steps_pgd     = 10
overshoot_df      = 0.02
max_iter_df       = 50

loss_fn_transfer = nn.CrossEntropyLoss()

# struttura risultati: una lista di dict, una riga per ogni (attacco, epsilon)
transfer_results = []

# esempi per i plot (un set per attacco)
examples_store = {
    'FGSM':    {'success': [], 'failure': []},
    'PGD':     {'success': [], 'failure': []},
    'DeepFool':{'success': [], 'failure': []},
}

source_model.eval()
target_model.eval()

# ---- FGSM Transfer ----
print("=" * 50)
print("FGSM Transfer Attack")
print("=" * 50)

for epsilon in epsilons_transfer:
    print(f"\n  epsilon = {epsilon}")

    source_clean_correct = 0
    target_clean_correct = 0
    source_adv_correct   = 0
    target_adv_correct   = 0
    both_correct_count   = 0
    transfer_success_count = 0
    whitebox_success_count = 0
    total = 0

    l2_list   = []
    linf_list = []

    start_time = time.time()

    for images, labels in tqdm(test_loader, desc=f"FGSM eps={epsilon}"):
        images, labels = images.to(device), labels.to(device)

        # predizioni clean
        with torch.no_grad():
            source_clean_preds = torch.argmax(source_model(NORMALIZE(images)), dim=1)
            target_clean_preds = torch.argmax(target_model(NORMALIZE(images)), dim=1)

        # genera adv sul SOURCE (ResNet18) — usa gradienti del source
        adv_images, perturbation = fgsm_attack(
            source_model,
            images.clone().requires_grad_(True),
            labels,
            epsilon,
            loss_fn_transfer
        )

        # valuta adv sul TARGET (MobileNetV2) — solo forward, no gradienti
        with torch.no_grad():
            source_adv_preds = torch.argmax(source_model(NORMALIZE(adv_images)), dim=1)
            target_adv_preds = torch.argmax(target_model(NORMALIZE(adv_images)), dim=1)

        both_correct    = (source_clean_preds == labels) & (target_clean_preds == labels)
        transfer_success = both_correct & (target_adv_preds != labels)
        whitebox_success = (source_clean_preds == labels) & (source_adv_preds != labels)

        source_clean_correct   += (source_clean_preds == labels).sum().item()
        target_clean_correct   += (target_clean_preds == labels).sum().item()
        source_adv_correct     += (source_adv_preds   == labels).sum().item()
        target_adv_correct     += (target_adv_preds   == labels).sum().item()
        both_correct_count     += both_correct.sum().item()
        transfer_success_count += transfer_success.sum().item()
        whitebox_success_count += whitebox_success.sum().item()
        total                  += labels.size(0)

        l2_norms   = perturbation.reshape(perturbation.size(0), -1).norm(p=2,   dim=1)
        linf_norms = perturbation.reshape(perturbation.size(0), -1).norm(p=float('inf'), dim=1)
        l2_list.extend(l2_norms.cpu().numpy())
        linf_list.extend(linf_norms.cpu().numpy())

        # raccoglie esempi per il plot (solo per epsilon=0.01)
        if epsilon == 0.01:
            for i in range(images.size(0)):
                s_clean = source_clean_preds[i].item()
                t_clean = target_clean_preds[i].item()
                t_adv   = target_adv_preds[i].item()
                true    = labels[i].item()
                ex = {
                    'image':        images[i:i+1].detach().cpu(),
                    'adv_image':    adv_images[i:i+1].detach().cpu(),
                    'perturbation': perturbation[i:i+1].detach().cpu(),
                    'source_clean': s_clean,
                    'target_clean': t_clean,
                    'target_adv':   t_adv,
                    'true_label':   true,
                }
                if s_clean == true and t_clean == true and t_adv != true:
                    examples_store['FGSM']['success'].append(ex)
                elif s_clean == true and t_clean == true and t_adv == true:
                    examples_store['FGSM']['failure'].append(ex)

    elapsed = time.time() - start_time

    source_clean_acc = source_clean_correct / total
    target_clean_acc = target_clean_correct / total
    source_adv_acc   = source_adv_correct   / total
    target_adv_acc   = target_adv_correct   / total
    whitebox_asr     = whitebox_success_count / source_clean_correct if source_clean_correct > 0 else 0
    transfer_asr     = transfer_success_count / both_correct_count   if both_correct_count   > 0 else 0
    mean_l2          = float(np.mean(l2_list))
    mean_linf        = float(np.mean(linf_list))

    print(f"    Source clean acc : {source_clean_acc:.4f}")
    print(f"    Target clean acc : {target_clean_acc:.4f}")
    print(f"    Source adv acc   : {source_adv_acc:.4f}")
    print(f"    Target adv acc   : {target_adv_acc:.4f}")
    print(f"    White-box ASR    : {whitebox_asr:.4f}")
    print(f"    Transfer ASR     : {transfer_asr:.4f}")
    print(f"    Mean L2          : {mean_l2:.4f}")
    print(f"    Mean Linf        : {mean_linf:.4f}")
    print(f"    Tempo            : {elapsed:.2f}s")

    transfer_results.append({
        'attack':           'FGSM',
        'epsilon':          epsilon,
        'source_clean_acc': source_clean_acc,
        'target_clean_acc': target_clean_acc,
        'source_adv_acc':   source_adv_acc,
        'target_adv_acc':   target_adv_acc,
        'whitebox_asr':     whitebox_asr,
        'transfer_asr':     transfer_asr,
        'mean_l2':          mean_l2,
        'mean_linf':        mean_linf,
        'time_s':           elapsed,
    })

print("\nFGSM Transfer completato")


In [ ]:
# ---- PGD Transfer ----
print("=" * 50)
print("PGD Transfer Attack")
print("=" * 50)

for epsilon in epsilons_transfer:
    alpha = epsilon / 4
    print(f"\n  epsilon = {epsilon}, alpha = {alpha:.5f}, steps = {num_steps_pgd}")

    source_clean_correct = 0
    target_clean_correct = 0
    source_adv_correct   = 0
    target_adv_correct   = 0
    both_correct_count   = 0
    transfer_success_count = 0
    whitebox_success_count = 0
    total = 0

    l2_list   = []
    linf_list = []

    start_time = time.time()

    for images, labels in tqdm(test_loader, desc=f"PGD eps={epsilon}"):
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            source_clean_preds = torch.argmax(source_model(NORMALIZE(images)), dim=1)
            target_clean_preds = torch.argmax(target_model(NORMALIZE(images)), dim=1)

        adv_images, perturbation = pgd_attack(
            source_model,
            images.clone().detach(),
            labels,
            epsilon,
            alpha,
            num_steps_pgd,
            loss_fn_transfer
        )

        with torch.no_grad():
            source_adv_preds = torch.argmax(source_model(NORMALIZE(adv_images)), dim=1)
            target_adv_preds = torch.argmax(target_model(NORMALIZE(adv_images)), dim=1)

        both_correct     = (source_clean_preds == labels) & (target_clean_preds == labels)
        transfer_success = both_correct & (target_adv_preds != labels)
        whitebox_success = (source_clean_preds == labels) & (source_adv_preds != labels)

        source_clean_correct   += (source_clean_preds == labels).sum().item()
        target_clean_correct   += (target_clean_preds == labels).sum().item()
        source_adv_correct     += (source_adv_preds   == labels).sum().item()
        target_adv_correct     += (target_adv_preds   == labels).sum().item()
        both_correct_count     += both_correct.sum().item()
        transfer_success_count += transfer_success.sum().item()
        whitebox_success_count += whitebox_success.sum().item()
        total                  += labels.size(0)

        l2_norms   = perturbation.reshape(perturbation.size(0), -1).norm(p=2,   dim=1)
        linf_norms = perturbation.reshape(perturbation.size(0), -1).norm(p=float('inf'), dim=1)
        l2_list.extend(l2_norms.cpu().numpy())
        linf_list.extend(linf_norms.cpu().numpy())

        if epsilon == 0.01:
            for i in range(images.size(0)):
                s_clean = source_clean_preds[i].item()
                t_clean = target_clean_preds[i].item()
                t_adv   = target_adv_preds[i].item()
                true    = labels[i].item()
                ex = {
                    'image':        images[i:i+1].detach().cpu(),
                    'adv_image':    adv_images[i:i+1].detach().cpu(),
                    'perturbation': perturbation[i:i+1].detach().cpu(),
                    'source_clean': s_clean,
                    'target_clean': t_clean,
                    'target_adv':   t_adv,
                    'true_label':   true,
                }
                if s_clean == true and t_clean == true and t_adv != true:
                    examples_store['PGD']['success'].append(ex)
                elif s_clean == true and t_clean == true and t_adv == true:
                    examples_store['PGD']['failure'].append(ex)

    elapsed = time.time() - start_time

    source_clean_acc = source_clean_correct / total
    target_clean_acc = target_clean_correct / total
    source_adv_acc   = source_adv_correct   / total
    target_adv_acc   = target_adv_correct   / total
    whitebox_asr     = whitebox_success_count / source_clean_correct if source_clean_correct > 0 else 0
    transfer_asr     = transfer_success_count / both_correct_count   if both_correct_count   > 0 else 0
    mean_l2          = float(np.mean(l2_list))
    mean_linf        = float(np.mean(linf_list))

    print(f"    Source clean acc : {source_clean_acc:.4f}")
    print(f"    Target clean acc : {target_clean_acc:.4f}")
    print(f"    Source adv acc   : {source_adv_acc:.4f}")
    print(f"    Target adv acc   : {target_adv_acc:.4f}")
    print(f"    White-box ASR    : {whitebox_asr:.4f}")
    print(f"    Transfer ASR     : {transfer_asr:.4f}")
    print(f"    Mean L2          : {mean_l2:.4f}")
    print(f"    Mean Linf        : {mean_linf:.4f}")
    print(f"    Tempo            : {elapsed:.2f}s")

    transfer_results.append({
        'attack':           'PGD',
        'epsilon':          epsilon,
        'source_clean_acc': source_clean_acc,
        'target_clean_acc': target_clean_acc,
        'source_adv_acc':   source_adv_acc,
        'target_adv_acc':   target_adv_acc,
        'whitebox_asr':     whitebox_asr,
        'transfer_asr':     transfer_asr,
        'mean_l2':          mean_l2,
        'mean_linf':        mean_linf,
        'time_s':           elapsed,
    })

print("\nPGD Transfer completato")


In [ ]:
# ---- DeepFool Transfer ----
print("=" * 50)
print("DeepFool Transfer Attack")
print("=" * 50)
print(f"overshoot={overshoot_df}, max_iter={max_iter_df}")

source_clean_correct = 0
target_clean_correct = 0
source_adv_correct   = 0
target_adv_correct   = 0
both_correct_count   = 0
transfer_success_count = 0
whitebox_success_count = 0
total = 0

l2_list   = []
linf_list = []

start_time = time.time()

for images, labels in tqdm(test_loader, desc="DeepFool Transfer"):
    images, labels = images.to(device), labels.to(device)

    with torch.no_grad():
        source_clean_preds = torch.argmax(source_model(NORMALIZE(images)), dim=1)
        target_clean_preds = torch.argmax(target_model(NORMALIZE(images)), dim=1)

    adv_images, perturbation = deepfool_attack(
        source_model,
        images.clone().detach(),
        N_CLASSES,
        overshoot_df,
        max_iter_df
    )

    with torch.no_grad():
        source_adv_preds = torch.argmax(source_model(NORMALIZE(adv_images)), dim=1)
        target_adv_preds = torch.argmax(target_model(NORMALIZE(adv_images)), dim=1)

    both_correct     = (source_clean_preds == labels) & (target_clean_preds == labels)
    transfer_success = both_correct & (target_adv_preds != labels)
    whitebox_success = (source_clean_preds == labels) & (source_adv_preds != labels)

    source_clean_correct   += (source_clean_preds == labels).sum().item()
    target_clean_correct   += (target_clean_preds == labels).sum().item()
    source_adv_correct     += (source_adv_preds   == labels).sum().item()
    target_adv_correct     += (target_adv_preds   == labels).sum().item()
    both_correct_count     += both_correct.sum().item()
    transfer_success_count += transfer_success.sum().item()
    whitebox_success_count += whitebox_success.sum().item()
    total                  += labels.size(0)

    l2_norms   = perturbation.reshape(perturbation.size(0), -1).norm(p=2,   dim=1)
    linf_norms = perturbation.reshape(perturbation.size(0), -1).norm(p=float('inf'), dim=1)
    l2_list.extend(l2_norms.cpu().numpy())
    linf_list.extend(linf_norms.cpu().numpy())

    for i in range(images.size(0)):
        s_clean = source_clean_preds[i].item()
        t_clean = target_clean_preds[i].item()
        t_adv   = target_adv_preds[i].item()
        true    = labels[i].item()
        ex = {
            'image':        images[i:i+1].detach().cpu(),
            'adv_image':    adv_images[i:i+1].detach().cpu(),
            'perturbation': perturbation[i:i+1].detach().cpu(),
            'source_clean': s_clean,
            'target_clean': t_clean,
            'target_adv':   t_adv,
            'true_label':   true,
            'l2_norm':      l2_norms[i].item(),
        }
        if s_clean == true and t_clean == true and t_adv != true:
            examples_store['DeepFool']['success'].append(ex)
        elif s_clean == true and t_clean == true and t_adv == true:
            examples_store['DeepFool']['failure'].append(ex)

elapsed = time.time() - start_time

source_clean_acc = source_clean_correct / total
target_clean_acc = target_clean_correct / total
source_adv_acc   = source_adv_correct   / total
target_adv_acc   = target_adv_correct   / total
whitebox_asr     = whitebox_success_count / source_clean_correct if source_clean_correct > 0 else 0
transfer_asr     = transfer_success_count / both_correct_count   if both_correct_count   > 0 else 0
mean_l2          = float(np.mean(l2_list))
mean_linf        = float(np.mean(linf_list))

print(f"  Source clean acc : {source_clean_acc:.4f}")
print(f"  Target clean acc : {target_clean_acc:.4f}")
print(f"  Source adv acc   : {source_adv_acc:.4f}")
print(f"  Target adv acc   : {target_adv_acc:.4f}")
print(f"  White-box ASR    : {whitebox_asr:.4f}")
print(f"  Transfer ASR     : {transfer_asr:.4f}")
print(f"  Mean L2          : {mean_l2:.4f}")
print(f"  Mean Linf        : {mean_linf:.4f}")
print(f"  Tempo            : {elapsed:.2f}s")

transfer_results.append({
    'attack':           'DeepFool',
    'epsilon':          overshoot_df,
    'source_clean_acc': source_clean_acc,
    'target_clean_acc': target_clean_acc,
    'source_adv_acc':   source_adv_acc,
    'target_adv_acc':   target_adv_acc,
    'whitebox_asr':     whitebox_asr,
    'transfer_asr':     transfer_asr,
    'mean_l2':          mean_l2,
    'mean_linf':        mean_linf,
    'time_s':           elapsed,
})

print("\nDeepFool Transfer completato")


## 6. Plot immagini perturbate (Transfer)

Per ogni attacco: originale / perturbazione / adversarial — stesso stile 3×4 dei notebook precedenti.
Le immagini mostrate sono quelle con epsilon=0.01 per FGSM/PGD e overshoot=0.02 per DeepFool.

In [ ]:
for attack_name, store in examples_store.items():
    selected = store['success'][:2] + store['failure'][:2]
    if len(selected) < 4:
        print(f"[{attack_name}] esempi insufficienti per il plot ({len(selected)}/4), skip")
        continue

    fig, axes = plt.subplots(3, 4, figsize=(15, 9))

    for img_idx in range(4):
        ex = selected[img_idx]

        img_orig = np.transpose(np.clip(ex['image'][0].numpy(), 0, 1), (1, 2, 0))
        pert     = np.transpose(ex['perturbation'][0].numpy(), (1, 2, 0))
        img_adv  = np.transpose(np.clip(ex['adv_image'][0].numpy(), 0, 1), (1, 2, 0))

        true_label   = ex['true_label']
        source_clean = ex['source_clean']
        target_clean = ex['target_clean']
        target_adv   = ex['target_adv']

        ax = axes[0, img_idx]
        ax.imshow(img_orig)
        ax.set_title(
            f"Originale\nTrue: {class_names[true_label]}\n"
            f"ResNet18: {class_names[source_clean]}\n"
            f"MobileNet: {class_names[target_clean]}",
            fontsize=9
        )
        ax.axis('off')

        ax = axes[1, img_idx]
        ax.imshow(np.abs(pert))
        ax.set_title(f"Perturbazione\n({attack_name})", fontsize=9)
        ax.axis('off')

        ax = axes[2, img_idx]
        ax.imshow(img_adv)
        color = 'red' if target_adv != true_label else 'green'
        ax.set_title(
            f"Adversarial\nTrue: {class_names[true_label]}\n"
            f"MobileNet: {class_names[target_adv]}",
            fontsize=9,
            color=color
        )
        ax.axis('off')

    plt.suptitle(f"Transfer Attack — {attack_name} (ResNet18 → MobileNetV2)", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(results_path / f"Transfer_{attack_name}_examples.png", dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Salvato: Transfer_{attack_name}_examples.png")


## 7. Metriche e salvataggio risultati

Tabelle riassuntive per FGSM, PGD e DeepFool transfer + plot comparativo White-Box ASR vs Transfer ASR.

In [ ]:
transfer_df = pd.DataFrame(transfer_results)

print("Tabella risultati Transfer Attack:")
print(transfer_df.to_string(index=False))

# salva per attacco
for attack_name in ['FGSM', 'PGD', 'DeepFool']:
    subset = transfer_df[transfer_df['attack'] == attack_name]
    subset.to_csv(results_path / f"Transfer_{attack_name}_metrics_summary.csv", index=False)

# salva tabella completa
transfer_df.to_csv(results_path / "Transfer_original_vs_adv.csv", index=False)

# ---- Plot comparativo White-Box ASR vs Transfer ASR ----
sns.set(style='whitegrid')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("White-Box ASR vs Transfer ASR (ResNet18 → MobileNetV2)", fontsize=13, fontweight='bold')

for ax, attack_name in zip(axes, ['FGSM', 'PGD', 'DeepFool']):
    subset = transfer_df[transfer_df['attack'] == attack_name].copy()
    subset = subset.sort_values('epsilon')

    x = np.arange(len(subset))
    width = 0.35

    bars1 = ax.bar(x - width/2, subset['whitebox_asr'], width, label='White-Box ASR', color='steelblue')
    bars2 = ax.bar(x + width/2, subset['transfer_asr'],  width, label='Transfer ASR',  color='coral')

    ax.set_xticks(x)
    ax.set_xticklabels([str(e) for e in subset['epsilon']], fontsize=9)
    ax.set_xlabel('Parametro principale', fontsize=10)
    ax.set_ylabel('ASR', fontsize=10)
    ax.set_title(attack_name, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(results_path / "Transfer_comparison_plot.png", dpi=120, bbox_inches='tight')
plt.show()
print("Salvato: Transfer_comparison_plot.png")
